# ML-09 — Validation and Research Claim Audit



In [14]:
!ls -la /content

total 20
drwxr-xr-x 1 root root 4096 Sep  2 14:47 .
drwxr-xr-x 1 root root 4096 Sep  2 13:39 ..
drwxr-xr-x 4 root root 4096 Aug 24 13:21 .config
drwx------ 5 root root 4096 Sep  2 14:47 drive
drwxr-xr-x 1 root root 4096 Aug 24 13:21 sample_data


In [16]:
%cd /content
!git clone https://github.com/Dev-hashh/flyrank-ml-internship-starter
%cd /content/flyrank-ml-internship-starter

/content
Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 170, done.
remote: Counting objects: 100% (170/170), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 170 (delta 74), reused 84 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (170/170), 1.96 MiB | 9.30 MiB/s, done.
Resolving deltas: 100% (74/74), done.
/content/flyrank-ml-internship-starter


In [17]:
!pwd
!ls

/content/flyrank-ml-internship-starter
AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


In [18]:
!python scripts/run_all.py


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship-starter/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship-starter/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship-starter/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship-starter/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /content/flyrank-ml-internship-starter/outputs/refresh_que

## 1. Two paper findings + my methodology questions

### Finding 1 — Refreshed older content can perform similarly to newer content

The paper reports that, in the analysed active-content sample, older content that had been recently refreshed performed similarly to some newer and freshly updated content groups. The paper also notes that parts of the comparison may be affected by survivor bias.

**My methodology question:**

I would ask how much of the observed difference can be attributed to the refresh itself rather than to differences that already existed between the pages. For example, were the older pages selected for refresh because they were already high-value or high-performing? If so, refreshed and non-refreshed pages may not be directly comparable.

A stronger validation design could compare similar pages before and after refresh, or compare refreshed pages with a matched group of similar pages that were not refreshed. This would not be a criticism of the paper's analysis; it is a constructive question about whether the observational comparison supports a causal interpretation.

Therefore, I would interpret the finding as an observed association: recently refreshed older content was associated with stronger measured performance in this sample. The analysis alone does not prove that refreshing caused the improvement.



### Finding 2 — AI-referred sessions showed strong growth over the observed months

The paper reports that AI-referred sessions increased across the portfolio during the analysed monthly period.

**My methodology question:**

I would ask whether the observed growth is entirely comparable across all months and clients. The number of active content pieces also increased over time, so part of the increase in AI sessions could reflect a larger observed content base rather than growth in AI referral performance per page.

I would also check whether tracking coverage changed during the period and whether the known-referral rules used to identify AI traffic remained consistent across all months.

A stronger validation would examine normalized measures such as AI sessions per active page or AI traffic share, while also checking whether the client mix and measurement coverage remained stable.

Therefore, I would describe this as a measured portfolio trend rather than proof that every page or client experienced the same AI-traffic growth.

## 2. My model under an honest split (before/after)


## Validation question

My Week-5 notebook already used a client-group holdout split. In this audit, I compare that design with a simpler random row split to understand whether validation design changes the measured performance.

### Before: random row split

A random row split can place pages from the same client in both the training and test sets. Pages from the same client may share content strategy, search patterns, audience characteristics, and measurement structure. This can make the test set easier because the model may learn client-specific patterns that are also present in testing.

### After: client-group holdout

In the grouped split, entire clients are held out. No client appears in both training and testing.

This is a stronger validation design for this dataset because the model is tested on clients it did not see during training.

The goal is not to obtain a higher score under the grouped split. The goal is to measure how the estimated model performance changes when the test design better matches the intended generalization problem.

In [19]:
# Setup, imports, and data loading

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")

def find_project_root():
    """Find the repo root in Jupyter or Colab without hard-coding a username/path."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, Path("/content")]

    # Common Colab case: the repo is one directory below /content.
    content_dir = Path("/content")
    if content_dir.exists():
        candidates.extend([p for p in content_dir.iterdir() if p.is_dir()])

    for candidate in candidates:
        if (
            (candidate / "data" / "processed" / "refresh_feature_vector.csv").exists()
            and (candidate / "data" / "processed" / "baseline_refresh_queue.csv").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not find the ML-07 processed files. Run the earlier pipeline first: "
        "`python scripts/run_all.py`, then run this notebook from the same repo."
    )


PROJECT_ROOT = find_project_root()
FEATURE_PATH = PROJECT_ROOT / "data/processed/refresh_feature_vector.csv"
BASELINE_PATH = PROJECT_ROOT / "data/processed/baseline_refresh_queue.csv"

features = pd.read_csv(FEATURE_PATH)
baseline = pd.read_csv(BASELINE_PATH)

print("Project root:", PROJECT_ROOT)
print("Feature data shape:", features.shape)
print("Baseline queue shape:", baseline.shape)
print("\nFeature columns:")
print(features.columns.tolist())

Project root: /content/flyrank-ml-internship-starter
Feature data shape: (30000, 52)
Baseline queue shape: (30000, 22)

Feature columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_s

In [22]:
features = pd.read_csv(FEATURE_PATH)
baseline = pd.read_csv(BASELINE_PATH)

# Build the modeling table

TARGET = "is_declining_label"
GROUP = "client_id"
ID_COLUMNS = ["content_id", "client_id"]

# Find the Week-4 baseline score column
score_candidates = [
    "baseline_refresh_score",
    "baseline_score",
]

baseline_score_col = next(
    (c for c in score_candidates if c in baseline.columns),
    None
)

if baseline_score_col is None:
    matching = [
        c for c in baseline.columns
        if "baseline" in c.lower() and "score" in c.lower()
    ]

    if matching:
        baseline_score_col = matching[0]
    else:
        raise KeyError("No baseline score column found.")

print("Baseline score column:", baseline_score_col)

# Create the baseline table used for merging
baseline_for_merge = (
    baseline[["content_id", baseline_score_col]]
    .drop_duplicates(subset="content_id")
)

# Merge features and baseline
df = features.merge(
    baseline_for_merge,
    on="content_id",
    how="inner",
    validate="one_to_one",
)

# Make target explicitly binary
df[TARGET] = df[TARGET].astype(int)

print("Rows after merge:", len(df))
print("Positive label rate:", f"{df[TARGET].mean():.3f}")
print("Number of clients:", df[GROUP].nunique())

df = features.merge(
    baseline_for_merge,
    on="content_id",
    how="inner",
    validate="one_to_one",
)

Baseline score column: baseline_refresh_score
Rows after merge: 30000
Positive label rate: 0.542
Number of clients: 32


In [25]:
LEAKAGE_COLUMNS = [
    TARGET,
    "trend_direction",
    "trend_pct",
    baseline_score_col,
    "reason_codes",
    "baseline_reason_codes",
    "final_refresh_score",
    "final_reason_codes",
]

numeric_candidates = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_candidates = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
]

numeric_features = [
    c for c in numeric_candidates if c in df.columns
]

categorical_features = [
    c for c in categorical_candidates if c in df.columns
]

model_features = numeric_features + categorical_features

for col in ID_COLUMNS + LEAKAGE_COLUMNS:
    if col in model_features:
        model_features.remove(col)

print("Number of model features:", len(model_features))
print(model_features)

Number of model features: 38
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']


In [26]:
# Target
TARGET = "is_declining_label"

# Features
X = df[model_features]

# Target values
y = df[TARGET]

# Client groups for grouped validation
groups = df["client_id"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of clients:", groups.nunique())

X shape: (30000, 38)
y shape: (30000,)
Number of clients: 32


In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split, GroupShuffleSplit

# BEFORE: Random row split

random_train_idx, random_test_idx = train_test_split(
    np.arange(len(X)),
    test_size=0.20,
    random_state=42,
    stratify=y,
)

X_train_random = X.iloc[random_train_idx]
X_test_random = X.iloc[random_test_idx]

y_train_random = y.iloc[random_train_idx]
y_test_random = y.iloc[random_test_idx]


# AFTER: Honest client-group holdout

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

group_train_idx, group_test_idx = next(
    group_splitter.split(X, y, groups=groups)
)

X_train_group = X.iloc[group_train_idx]
X_test_group = X.iloc[group_test_idx]

y_train_group = y.iloc[group_train_idx]
y_test_group = y.iloc[group_test_idx]


# Check client overlap
random_train_clients = set(groups.iloc[random_train_idx])
random_test_clients = set(groups.iloc[random_test_idx])

group_train_clients = set(groups.iloc[group_train_idx])
group_test_clients = set(groups.iloc[group_test_idx])

print("RANDOM ROW SPLIT")
print("Train rows:", len(X_train_random))
print("Test rows:", len(X_test_random))
print("Client overlap:", len(random_train_clients & random_test_clients))

print("\nCLIENT-GROUP HOLDOUT")
print("Train rows:", len(X_train_group))
print("Test rows:", len(X_test_group))
print("Train clients:", len(group_train_clients))
print("Test clients:", len(group_test_clients))
print("Client overlap:", len(group_train_clients & group_test_clients))

RANDOM ROW SPLIT
Train rows: 24000
Test rows: 6000
Client overlap: 31

CLIENT-GROUP HOLDOUT
Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0


In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

# ---------------------------------------
# Preprocessing
# ---------------------------------------

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore")
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


# ---------------------------------------
# Helper function
# ---------------------------------------

def evaluate_scores(y_true, scores, k=50):

    k = min(k, len(y_true))

    top_k_idx = np.argsort(scores)[::-1][:k]

    return {
        "ROC-AUC": roc_auc_score(y_true, scores),
        "Average Precision": average_precision_score(
            y_true,
            scores
        ),
        "Precision@50": np.mean(
            y_true.iloc[top_k_idx].values
        ),
    }


# ---------------------------------------
# Function to build the same model pipeline
# ---------------------------------------

def build_model():

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=14,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )


# ---------------------------------------
# BEFORE: Random row split
# ---------------------------------------

random_model = build_model()

random_model.fit(
    X_train_random,
    y_train_random
)

random_scores = random_model.predict_proba(
    X_test_random
)[:, 1]

random_results = evaluate_scores(
    y_test_random,
    random_scores
)


# ---------------------------------------
# AFTER: Client-group holdout
# ---------------------------------------

group_model = build_model()

group_model.fit(
    X_train_group,
    y_train_group
)

group_scores = group_model.predict_proba(
    X_test_group
)[:, 1]

group_results = evaluate_scores(
    y_test_group,
    group_scores
)


# ---------------------------------------
# Comparison table
# ---------------------------------------

validation_comparison = pd.DataFrame([
    {
        "Validation Design": "Random Row Split",
        **random_results
    },
    {
        "Validation Design": "Client-Group Holdout",
        **group_results
    },
])

display(
    validation_comparison.style.format({
        "ROC-AUC": "{:.3f}",
        "Average Precision": "{:.3f}",
        "Precision@50": "{:.3f}",
    })
)

,Validation Design,ROC-AUC,Average Precision,Precision@50
0,Random Row Split,0.913,0.928,1.000
1,Client-Group Holdout,0.844,0.854,0.980


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.